# 01 · Exploratory data analysis — MovieLens 100K

100,000 ratings (1–5 stars) from 943 users on 1,682 movies, collected by GroupLens in 1997–1998. Each user has at least 20 ratings.

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 4); pd.set_option("display.width", 140)
RESULTS = Path("../results"); RESULTS.mkdir(exist_ok=True)

def save_params(name, params):
    path = RESULTS / "best_params.json"
    allp = json.loads(path.read_text()) if path.exists() else {}
    allp[name] = {k: (v.item() if hasattr(v, "item") else v) for k, v in params.items()}
    path.write_text(json.dumps(allp, indent=2))

In [ ]:
from src.data import load_ratings, load_movies, load_users, temporal_user_split, GENRES
ratings, movies, users = load_ratings(), load_movies(), load_users()
ratings.head()

In [ ]:
n_u, n_i = ratings.user_idx.nunique(), ratings.item_idx.nunique()
print(f"users={n_u}, movies={n_i}, ratings={len(ratings)}")
print(f"density = {len(ratings) / (n_u * n_i):.4%}  (sparsity {1 - len(ratings) / (n_u * n_i):.2%})")
print("rating period:", pd.to_datetime(ratings.timestamp.min(), unit="s").date(), "→",
      pd.to_datetime(ratings.timestamp.max(), unit="s").date())
print("mean rating:", round(ratings.rating.mean(), 3))

## Rating distribution

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ratings.rating.value_counts().sort_index().plot.bar(ax=ax[0], title="Ratings", color="#4C72B0")
per_user = ratings.groupby("user_idx").size()
per_item = ratings.groupby("item_idx").size()
per_user.plot.hist(bins=50, ax=ax[1], title="Ratings per user", color="#4C72B0")
per_item.plot.hist(bins=50, ax=ax[2], title="Ratings per movie", color="#4C72B0")
ax[1].set_yscale("log"); ax[2].set_yscale("log"); plt.tight_layout()
pd.DataFrame({"per_user": per_user.describe(), "per_movie": per_item.describe()}).T

## Long tail

How concentrated are the ratings on popular movies?

In [ ]:
share = per_item.sort_values(ascending=False).cumsum() / len(ratings)
for top in [0.05, 0.1, 0.2]:
    k = int(top * n_i)
    print(f"top {top:.0%} of movies ({k}) receive {share.iloc[k - 1]:.1%} of all ratings")
print("movies with < 5 ratings:", int((per_item < 5).sum()))
plt.figure(figsize=(6, 3.5)); plt.plot(np.arange(1, n_i + 1) / n_i, share.values)
plt.xlabel("share of movies (sorted by popularity)"); plt.ylabel("share of ratings"); plt.grid(alpha=.3);

## Average rating vs. number of ratings

Movies with very few ratings have extreme averages — the reason the popularity model shrinks the mean.

In [ ]:
stats = ratings.groupby("item_idx").rating.agg(["count", "mean"])
plt.figure(figsize=(6, 4)); plt.scatter(stats["count"], stats["mean"], s=6, alpha=.4)
plt.xscale("log"); plt.xlabel("number of ratings (log)"); plt.ylabel("mean rating");
stats.join(movies.set_index("item_idx").title).sort_values("mean", ascending=False).head(8)

## Genres and users

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
movies[GENRES].sum().sort_values().plot.barh(ax=ax[0], title="Movies per genre", color="#4C72B0")
users.age.plot.hist(bins=30, ax=ax[1], title="User age", color="#4C72B0"); plt.tight_layout()
users.gender.value_counts(normalize=True)

## Train / test split and leakage checks

We use a **per-user temporal split**: for every user, the last 20% of their ratings (by timestamp) form the test set. Model selection uses a second split of the same kind inside the training set (train_in / valid), so the test set is touched once, at the very end (notebook 05).

In [ ]:
train, test = temporal_user_split(ratings, test_frac=0.2)
train_in, valid = temporal_user_split(train, test_frac=0.2)
print(f"train={len(train)}, test={len(test)}, train_in={len(train_in)}, valid={len(valid)}")

assert len(train.merge(test, on=["user_idx", "item_idx"])) == 0
gap = test.groupby("user_idx").timestamp.min() - train.groupby("user_idx").timestamp.max()
print("users whose train contains a rating later than their first test rating:", int((gap < 0).sum()))
print("users in test without train history:", len(set(test.user_idx) - set(train.user_idx)))
print("test ratings on movies never seen in train:", int((~test.item_idx.isin(train.item_idx)).sum()))
rel = test[test.rating >= 4].groupby("user_idx").size()
print(f"users with >=1 relevant (>=4 stars) test item: {len(rel)}; relevant per user: median {rel.median():.0f}")

**Remaining caveat.** The per-user split is chronological *within* a user, but the training set still contains ratings by other users that were made after a given user's test period. This is standard for ML-100K but optimistic; notebook 05 repeats the comparison on a strict global time split as a robustness check.